In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn

from mhealth_activity.recording import Recording
from mhealth_activity.types import Activity
from mhealth_activity import WatchLocation
from mhealth_activity import Trace

# Windowed & peak-based baseline #

Estimate steps from accelerometer magnitude using local peak detection.

Method (brief):

- Compute a_mag, remove mean, bandpass (≈0.7–3 Hz)
- Split into short windows (~10 s)
- Detect peaks per window (distance + prominence)
- Keep windows with plausible cadence (≈1.3–2.8 Hz)
- Sum peaks → step count

In [5]:
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np

# DON'T TOUCH the hyperparameters, they are tuned and won't get better.
def estimate_steps_windowed(
    rec,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    min_peak_rate_hz=1.3,
    min_std_threshold=0.08,
    min_final_steps_to_keep=80,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    total_steps = 0
    kept_windows = []

    for start in range(0, len(filt), win_len):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        # energy-based gating
        seg_std = float(np.std(segment))
        if seg_std < min_std_threshold:
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        if peak_rate_hz > min_peak_rate_hz:
            total_steps += len(peaks)
            kept_windows.append((start, end, len(peaks), peak_rate_hz, seg_std))

    # global cleanup for tiny false positives
    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {
        "steps_hat": int(total_steps),
        "filtered_signal": filt,
        "fs": fs,
        "kept_windows": kept_windows,
    }

submission

In [6]:
# --- step-count-only baseline submission ---
import re
from pathlib import Path
import pandas as pd

test_dir = Path("data/test")
submission_name = "submission_step_count.csv"

def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d+)\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse Id from filename: {path.name}")
    return int(match.group(1))

rows = []

for path in sorted(test_dir.glob("*.pkl")):
    rec = Recording(str(path))
    step_pred = int(estimate_steps_windowed(rec)["steps_hat"])

    rows.append({
        "Id": parse_trace_id(path),
        "watch_loc": -1,      # default placeholder
        "path_idx": -1,       # default placeholder
        "standing": -1,   # default placeholder
        "walking": -1,    # default placeholder
        "running": -1,    # default placeholder
        "cycling": -1,    # default placeholder
        "step_count": step_pred,
    })

submission_df = pd.DataFrame(rows).sort_values("Id")

# optional sanity checks
print(submission_df.head())
print(f"\nRows: {len(submission_df)}")
print(f"Step count min/max: {submission_df['step_count'].min()} / {submission_df['step_count'].max()}")

submission_df.to_csv(submission_name, index=False)
print(f"\nSaved submission to {submission_name}")

   Id  watch_loc  path_idx  standing  walking  running  cycling  step_count
0   0         -1        -1        -1       -1       -1       -1         975
1   1         -1        -1        -1       -1       -1       -1         253
2   2         -1        -1        -1       -1       -1       -1         376
3   3         -1        -1        -1       -1       -1       -1        1093
4   4         -1        -1        -1       -1       -1       -1        1012

Rows: 280
Step count min/max: 0 / 1430

Saved submission to submission_step_count.csv
